## START

In [9]:
# Loading the documents
# We'll use helper files from module 01 and this module.

# If you don't have them in your notebook directory, download them:

# PREFIX='https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main'

# !wget {PREFIX}/01-agentic-rag/code/ingest.py
# !wget {PREFIX}/01-agentic-rag/code/rag_helper.py
# !wget {PREFIX}/04-evaluation/code/evaluation_utils.py
# Then load the FAQ data:

from ingest import load_faq_data
documents = load_faq_data()
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [10]:
# We'll generate questions only for the LLM Zoomcamp FAQ. The full FAQ dataset contains documents from multiple courses. 
# Generating five questions for every document would take longer and cost more.

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

113

In [11]:
# We'll use these documents from now on so let's name them as documents

documents = documents_llm
# Each document already has an id field:

doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


## Generating questions with structured output

In [13]:
# Generating questions with structured output
# We use an LLM to generate questions for each document.

# This is the first time we're using structured output in the course. With structured output, we ask the LLM to return data 
# in a specific format instead of free-form text. For example, instead of getting a paragraph that contains questions, 
# we can ask for a Python object with a questions field.

# This is useful when code will process the output. The model returns the same structure every time. We can access the 
# generated questions directly instead of parsing text manually.

# We want the output as a list of strings, so we define that structure with a Pydantic model:

from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

# The instructions for the LLM:

data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

# We ask the LLM to use different wording from the original document. This makes the evaluation more realistic - real users 
# won't phrase their questions the same way as the FAQ.

# Call the LLM for one document:

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

# Prepare the document as JSON:

import json

user_prompt = json.dumps(doc)
# Create the messages:

messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

# The parsed object is available in response.output_parsed:

result = response.output_parsed

print(result)
# We can access the list directly:

print(result.questions)
# You should see 5 questions that relate to the first FAQ document.

questions=['I found this course late — can I still join it, and is there anything I need to do if I want a certificate?', 'Is it too late to start the course now, or can I still enroll even after it started?', 'If I join the course late, do I still qualify for the certificate as long as I submit the project on time?', 'Can I start the course after it has begun, and what are the rules for getting certified?', 'I just heard about the course — is it still open for new students, and when do I need to turn in the project for the certificate?']
['I found this course late — can I still join it, and is there anything I need to do if I want a certificate?', 'Is it too late to start the course now, or can I still enroll even after it started?', 'If I join the course late, do I still qualify for the certificate as long as I submit the project on time?', 'Can I start the course after it has begun, and what are the rules for getting certified?', 'I just heard about the course — is it still open for

In [14]:
# Reusable utilities
# We'll need this pattern again in other evaluation sections today, so we put it in a reusable helper.

# It contains helper functions we'll reuse in this module:

# llm_structured: calls the OpenAI API with structured output
# llm_structured_retry: retries structured-output calls when a request fails
# calc_price: calculates the price from token usage
# calc_total_price: calculates the total price from multiple usage objects
# map_progress: runs work in parallel and tracks progress. We'll use it in the next lesson.
# Import the structured-output helper:

from evaluation_utils import llm_structured
# Use it on the same document:

result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['I just found this course late — is it still okay to join now?', 'If I join after the course has already started, can I still get the certificate?', 'Am I allowed to enroll even though the course is already underway?', 'What do I need to do to make sure I’m eligible for the course certificate if I start now?', 'If I’m joining late, is submitting the final project before the deadline enough for the certificate?']


## Tracking cost, parallel processing and ground truth questions generation

In [15]:
# Tracking cost
# The response also contains token usage:

# usage.input_tokens, usage.output_tokens
# As in the agents module, we calculate the price from response.usage.

# Import the price helper:

from evaluation_utils import calc_price
# Calculate the cost of this call:

cost = calc_price(usage)

cost

{'input_cost': 0.00015525, 'output_cost': 0.0004545, 'total_cost': 0.00060975}

In [16]:
# Now convert these questions into ground truth records:

records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

# Each record has two fields:

# question: the question generated by the LLM
# document: the ID of the FAQ document that should answer the question
# The document field connects the generated question to the document that contains the answer. 
# Later, when we evaluate search, we'll ask the search engine the generated question. Then we'll check if it retrieves the document with this ID.

# We now know how to generate and store questions for one document. 


[{'question': 'I just found this course late — is it still okay to join now?',
  'document': '74eb249bbf'},
 {'question': 'If I join after the course has already started, can I still get the certificate?',
  'document': '74eb249bbf'},
 {'question': 'Am I allowed to enroll even though the course is already underway?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to make sure I’m eligible for the course certificate if I start now?',
  'document': '74eb249bbf'},
 {'question': 'If I’m joining late, is submitting the final project before the deadline enough for the certificate?',
  'document': '74eb249bbf'}]

In [17]:
# Above we generated questions for one document and converted them into ground truth records.

# We want to do the same thing for every document in the FAQ dataset. For each document, we generate questions and save them as ground truth records.

# For this part, we'll use tqdm for progress bars and pandas for saving the final CSV.

# If you don't have them installed yet, add them first:

# uv add tqdm pandas
# The processing function takes one document and turns it into ground truth records.

# For each document, we:

# convert the document to JSON so we can send it to the LLM
# ask the LLM to return a Questions object
# create one ground truth record for each generated question
# Each record contains the generated question and the ID of the document that should answer the question.

# When we send many requests, one of them might fail. We don't want the entire batch to fail because of one temporary error.

# Import the retry helper from evaluation_utils.py:

from evaluation_utils import llm_structured_retry
# llm_structured makes one structured-output call. llm_structured_retry wraps the same call in a retry loop. 
# If one request fails because of a temporary API or network issue, it waits briefly and tries again.

# Use it in the processing function:

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage
    
# Try it for the first 5 documents.

# Import tqdm and run the loop:

from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)
    
# This works, but it runs one LLM call after another. Running it for all documents this way would take too long.

  0%|          | 0/5 [00:00<?, ?it/s]

In [18]:
ground_truth[10]

{'question': "How do I join the Office Hours or live workshop if the Zoom link isn't shared with students?",
 'document': '489dd1c9d9'}

In [19]:
len(ground_truth)

25

In [20]:
# Parallel processing
# Running the calls one after another wastes most of the time waiting on the network. 
# Each request just sits there until OpenAI responds, so we can fire several at once and wait on them together. 
# We process the documents in parallel and track progress while the requests run.

# One caution: don't open too many connections at once, or you'll hit the provider's rate limits. Five or six workers is a safe default here.

# Import ThreadPoolExecutor:

from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

# This submits one job per document, updates the progress bar when a job finishes, and collects the results. 
# If you want a more detailed explanation of ThreadPoolExecutor and futures, ask ChatGPT to walk through this helper line by line.

# Then replace the loop with the parallel version:

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)
    
# generate_ground_truth returns two things for each document: the generated records and the token usage.

  0%|          | 0/113 [00:00<?, ?it/s]

In [21]:
# Split those into separate lists:

ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

# With 5 questions per document, you should get roughly 5x the number of documents.

# Calculate the total cost:

from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.08631150000000003

In [23]:
# Create a dataframe so we can look at the records as a table and save them as a CSV file.

# Create the dataframe:

import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

# Because we generated the questions from specific documents, we know which document is correct for each question. 
# We now have the ground truth we need for evaluation.

# Save it for later use:

df_ground_truth.to_csv("ground_truth-new.csv", index=False)
    
# The FAQ data can change over time. If you run the notebook later, you may see different documents and generated questions. 
# Token usage, cost, and search evaluation results may also change.

# The total cost was 0.0863, about 9 cents.